# ✈️ Airline Passenger Satisfaction Analytics & Prediction

**AICTE | IBM SkillsBuild Data Analytics with AI Internship**

---

## 1. Project Introduction

This notebook presents a complete, end-to-end Data Analytics and Predictive Analytics project using the **Airline Passenger Satisfaction** dataset. The project follows the internship curriculum, covering:

- Data understanding and the data dictionary
- Data cleaning with justified decisions
- Exploratory Data Analysis (EDA)
- Business-question-driven analysis and storytelling
- Passenger segmentation
- Logistic Regression for binary satisfaction prediction
- Data leakage prevention
- Model evaluation (Confusion Matrix, Classification Report)
- Translating results into business recommendations

---

## 2. Business Objectives

| # | Objective |
|---|-----------|
| 1 | Understand the profile of satisfied vs. neutral/dissatisfied passengers |
| 2 | Identify which service dimensions are most strongly associated with satisfaction |
| 3 | Determine how passenger characteristics (type, travel purpose, class, age) affect satisfaction |
| 4 | Analyse operational factors (delays, flight distance) and their relationship with satisfaction |
| 5 | Build a predictive model (Logistic Regression) to classify passenger satisfaction |
| 6 | Provide actionable, data-driven business recommendations |

---

> **Dataset source:** The dataset used is `airline_passenger_satisfaction.csv`, provided for the AICTE | IBM SkillsBuild Data Analytics with AI Internship.  
> **Data Dictionary source:** `data_dictionary.csv` (provided alongside the dataset).


In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ── Data manipulation ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve
)

# ── Display settings ──────────────────────────────────────────────────────────
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_style('whitegrid')
sns.set_palette('Set2')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.labelsize': 11})

print("All libraries imported successfully.")


## 3. Dataset Loading

In [ ]:
# Load the raw dataset (never modify the original)
RAW_PATH = 'airline_passenger_satisfaction.csv'
DICT_PATH = 'data_dictionary.csv'
PROCESSED_PATH = 'data/processed_airline_passenger_data.csv'

df_raw = pd.read_csv(RAW_PATH)
df_dict = pd.read_csv(DICT_PATH)

print(f"Raw dataset loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Data dictionary loaded: {df_dict.shape[0]} fields documented")


## 4. Data Dictionary Interpretation

In [ ]:
# Display the enriched data dictionary
# We extend the provided data_dictionary.csv with analytical metadata
dict_extended = pd.DataFrame({
    'Field': df_raw.columns,
    'Description': [
        'Unique passenger identifier',
        'Gender of the passenger (Female/Male)',
        'Age of the passenger',
        'Type of airline customer (First-time/Returning)',
        'Purpose of the flight (Business/Personal)',
        'Travel class (Business/Economy/Economy Plus)',
        'Flight distance in miles',
        'Flight departure delay in minutes',
        'Flight arrival delay in minutes',
        'Departure/arrival time convenience rating (0=N/A, 1–5)',
        'Ease of online booking rating (0=N/A, 1–5)',
        'Check-in service rating (0=N/A, 1–5)',
        'Online boarding rating (0=N/A, 1–5)',
        'Gate location rating (0=N/A, 1–5)',
        'On-board service rating (0=N/A, 1–5)',
        'Seat comfort rating (0=N/A, 1–5)',
        'Leg room service rating (0=N/A, 1–5)',
        'Cleanliness rating (0=N/A, 1–5)',
        'Food and drink rating (0=N/A, 1–5)',
        'In-flight service rating (0=N/A, 1–5)',
        'In-flight Wi-Fi service rating (0=N/A, 1–5)',
        'In-flight entertainment rating (0=N/A, 1–5)',
        'Baggage handling rating (0=N/A, 1–5)',
        'Overall satisfaction (Satisfied / Neutral or Dissatisfied)',
    ],
    'Data Type': [str(t) for t in df_raw.dtypes],
    'Analytical Role': [
        'Identifier – excluded from modelling',
        'Demographic feature',
        'Demographic feature',
        'Segment feature',
        'Segment feature',
        'Segment feature',
        'Operational feature',
        'Operational feature',
        'Operational feature',
        'Service-quality feature',
        'Service-quality feature',
        'Service-quality feature',
        'Service-quality feature',
        'Service-quality feature',
        'Service-quality feature',
        'Service-quality feature',
        'Service-quality feature',
        'Service-quality feature',
        'Service-quality feature',
        'Service-quality feature',
        'Service-quality feature',
        'Service-quality feature',
        'Service-quality feature',
        'TARGET – binary classification',
    ],
    'Used in Model': [
        'No (ID)',
        'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes',
        'Yes','Yes','Yes','Yes','Yes','Yes','Yes','Yes','Yes','Yes',
        'Yes','Yes','Yes','Yes',
        'No (target)',
    ]
})
dict_extended


## 5. Data Audit

In [ ]:
# ── 5.1 Basic Shape ──────────────────────────────────────────────────────────
print("=" * 55)
print(f"  Rows    : {df_raw.shape[0]:>10,}")
print(f"  Columns : {df_raw.shape[1]:>10}")
print("=" * 55)

# ── 5.2 Data types ────────────────────────────────────────────────────────────
print("\nData types:")
print(df_raw.dtypes)


In [ ]:
# ── 5.3 Missing Values ────────────────────────────────────────────────────────
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(3)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]
print("Columns with missing values:")
print(missing_df if not missing_df.empty else "  None")


In [ ]:
# ── 5.4 Duplicate Rows ───────────────────────────────────────────────────────
dup_count = df_raw.duplicated().sum()
print(f"Duplicate rows: {dup_count}")

# ── 5.5 Target Distribution ──────────────────────────────────────────────────
print("\nTarget variable distribution:")
tgt = df_raw['Satisfaction'].value_counts()
print(tgt)
print(f"\nSatisfied rate : {tgt['Satisfied'] / len(df_raw) * 100:.2f}%")
print(f"Neutral/Dissatisfied rate : {tgt['Neutral or Dissatisfied'] / len(df_raw) * 100:.2f}%")


In [ ]:
# ── 5.6 Categorical Unique Values ────────────────────────────────────────────
cat_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class', 'Satisfaction']
for col in cat_cols:
    print(f"{col}: {df_raw[col].unique().tolist()}")


In [ ]:
# ── 5.7 Numerical Summary ────────────────────────────────────────────────────
num_summary = df_raw.describe().T
num_summary['range'] = num_summary['max'] - num_summary['min']
print("Numerical summary (key columns):")
print(num_summary[['count','mean','std','min','25%','50%','75%','max','range']].round(2))


In [ ]:
# ── 5.8 Service Rating Zero Counts (0 = Not Applicable) ─────────────────────
SERVICE_COLS = [
    'Departure and Arrival Time Convenience', 'Ease of Online Booking',
    'Check-in Service', 'Online Boarding', 'Gate Location',
    'On-board Service', 'Seat Comfort', 'Leg Room Service',
    'Cleanliness', 'Food and Drink', 'In-flight Service',
    'In-flight Wifi Service', 'In-flight Entertainment', 'Baggage Handling'
]
print("Zero-rating counts (0 = Not Applicable per data dictionary):")
zero_counts = {c: (df_raw[c] == 0).sum() for c in SERVICE_COLS}
for col, cnt in zero_counts.items():
    if cnt > 0:
        pct = cnt / len(df_raw) * 100
        print(f"  {col:<45}: {cnt:>5} ({pct:.2f}%)")

print("\nNOTE: Per the data dictionary, 0 ratings represent 'Not Applicable'.")
print("      They are NOT converted to NaN — they carry meaningful information.")


In [ ]:
# ── 5.9 Outlier Check: Delay Variables ───────────────────────────────────────
for col in ['Departure Delay', 'Arrival Delay']:
    q99 = df_raw[col].quantile(0.99)
    extreme = (df_raw[col] > q99).sum()
    print(f"{col}: 99th pct = {q99:.0f} min | values above 99th pct = {extreme:,}")


## 6. Data Cleaning

### Key Cleaning Decisions

| Issue | Decision | Justification |
|-------|----------|---------------|
| `ID` column | Excluded from modelling | Arbitrary identifier — carries no predictive signal |
| 393 missing `Arrival Delay` values | Imputed with median of `Departure Delay` per row (fallback: column median) | Arrival delay is highly correlated with departure delay; median imputation is conservative and preserves row count |
| Rating value = 0 | **Kept as-is** | Per data dictionary, 0 = "Not Applicable" — a meaningful categorical value |
| Duplicate rows | Dropped | Exact duplicates add no information |
| Categorical inconsistencies | None found | All categories match data dictionary |
| Negative delays | None found | Min delay = 0 minutes |


In [ ]:
# Start from a copy — never modify the original
df = df_raw.copy()

# ── 6.1 Drop duplicates ───────────────────────────────────────────────────────
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Duplicates removed: {before - len(df)}")

# ── 6.2 Impute missing Arrival Delay ─────────────────────────────────────────
# Strategy: fill each missing Arrival Delay with that row's Departure Delay
# (both represent operational delay; they are highly correlated).
# Rows where Departure Delay is also missing fall back to the column median.
arrival_median = df['Arrival Delay'].median()
df['Arrival Delay'] = df.apply(
    lambda row: row['Departure Delay']
        if pd.isna(row['Arrival Delay'])
        else row['Arrival Delay'],
    axis=1
)
# Any remaining NaNs (none expected, but safe fallback)
df['Arrival Delay'].fillna(arrival_median, inplace=True)
print(f"Arrival Delay missing after imputation: {df['Arrival Delay'].isnull().sum()}")

# ── 6.3 Drop ID (identifier — not a predictive feature) ──────────────────────
df.drop(columns=['ID'], inplace=True)
print(f"'ID' column removed. Remaining columns: {len(df.columns)}")

# ── 6.4 Encode target as binary integer (for modelling) ──────────────────────
df['Satisfaction_Binary'] = (df['Satisfaction'] == 'Satisfied').astype(int)
# 1 = Satisfied, 0 = Neutral or Dissatisfied

# ── 6.5 Create Age Group feature ─────────────────────────────────────────────
bins   = [0, 17, 29, 44, 59, 100]
labels = ['Under 18', '18–29', '30–44', '45–59', '60+']
df['Age Group'] = pd.cut(df['Age'], bins=bins, labels=labels)
print("Age Group distribution:")
print(df['Age Group'].value_counts().sort_index())

# ── 6.6 Save processed dataset ───────────────────────────────────────────────
df.to_csv(PROCESSED_PATH, index=False)
print(f"\nProcessed dataset saved → {PROCESSED_PATH}")
print(f"Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")


## 7. Exploratory Data Analysis (EDA)

### 7.1 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Count plot
order = ['Neutral or Dissatisfied', 'Satisfied']
counts = df['Satisfaction'].value_counts().reindex(order)
axes[0].bar(order, counts.values, color=['#e07b54', '#4c9ed9'], edgecolor='white', linewidth=0.5)
axes[0].set_title('Passenger Satisfaction Distribution')
axes[0].set_ylabel('Number of Passengers')
axes[0].set_xlabel('Satisfaction Level')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 300, f'{v:,}', ha='center', fontsize=10)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Pie chart
axes[1].pie(
    counts.values, labels=order, autopct='%1.1f%%',
    colors=['#e07b54', '#4c9ed9'], startangle=90,
    wedgeprops={'linewidth': 1.5, 'edgecolor': 'white'}
)
axes[1].set_title('Satisfaction Share')

plt.tight_layout()
plt.savefig('assets/charts/01_satisfaction_distribution.png', bbox_inches='tight')
plt.show()
print(f"Satisfied: {counts['Satisfied']:,} ({counts['Satisfied']/len(df)*100:.1f}%)")
print(f"Neutral or Dissatisfied: {counts['Neutral or Dissatisfied']:,} ({counts['Neutral or Dissatisfied']/len(df)*100:.1f}%)")


### 7.2 Satisfaction by Key Segments

In [ ]:
def stacked_sat_bar(col, ax, title):
    """Helper: stacked bar chart of satisfaction by a categorical column."""
    ct = pd.crosstab(df[col], df['Satisfaction'], normalize='index') * 100
    ct = ct[['Satisfied', 'Neutral or Dissatisfied']]
    ct.plot(kind='bar', stacked=True, ax=ax,
            color=['#4c9ed9', '#e07b54'], edgecolor='white', linewidth=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Percentage (%)')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)
    ax.legend(loc='lower right', fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
    for bar in ax.patches:
        h = bar.get_height()
        if h > 5:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_y() + h/2,
                    f'{h:.0f}%', ha='center', va='center', fontsize=7, color='white')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
stacked_sat_bar('Customer Type', axes[0], 'By Customer Type')
stacked_sat_bar('Type of Travel', axes[1], 'By Type of Travel')
stacked_sat_bar('Class', axes[2], 'By Travel Class')
plt.suptitle('Satisfaction by Passenger Segments', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('assets/charts/02_satisfaction_by_segment.png', bbox_inches='tight')
plt.show()


### 7.3 Satisfaction by Age Group

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

stacked_sat_bar('Age Group', axes[0], 'Satisfaction by Age Group')

# Age distribution by satisfaction
df.boxplot(column='Age', by='Satisfaction', ax=axes[1],
           boxprops=dict(color='#4c9ed9'),
           medianprops=dict(color='#e07b54', linewidth=2),
           whiskerprops=dict(color='#4c9ed9'),
           capprops=dict(color='#4c9ed9'))
axes[1].set_title('Age Distribution by Satisfaction')
axes[1].set_xlabel('Satisfaction Level')
axes[1].set_ylabel('Age (years)')
plt.suptitle('')
plt.tight_layout()
plt.savefig('assets/charts/03_age_satisfaction.png', bbox_inches='tight')
plt.show()

# Numerical summary
print("Median age by satisfaction group:")
print(df.groupby('Satisfaction')['Age'].median())


### 7.4 Service Dimension Ratings vs Satisfaction

In [ ]:
# Mean rating for each service column, split by satisfaction class
mean_ratings = df.groupby('Satisfaction')[SERVICE_COLS].mean()
diff = mean_ratings.loc['Satisfied'] - mean_ratings.loc['Neutral or Dissatisfied']
diff_sorted = diff.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
colours = ['#4c9ed9' if v >= 0 else '#e07b54' for v in diff_sorted.values]
ax.barh(diff_sorted.index, diff_sorted.values, color=colours, edgecolor='white')
ax.axvline(0, color='grey', linewidth=0.8, linestyle='--')
ax.set_title('Mean Rating Difference: Satisfied minus Neutral/Dissatisfied', fontweight='bold')
ax.set_xlabel('Mean Rating Difference (Satisfied − Neutral/Dissatisfied)')
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.savefig('assets/charts/04_service_rating_diff.png', bbox_inches='tight')
plt.show()

print("Top 5 service dimensions where Satisfied passengers rate higher:")
print(diff_sorted.head(5).round(3))


### 7.5 Correlation Heatmap (Service Ratings)

In [ ]:
corr_data = df[SERVICE_COLS + ['Satisfaction_Binary']].copy()
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu',
    center=0, linewidths=0.5, ax=ax, annot_kws={'size': 7},
    cbar_kws={'shrink': 0.6}
)
ax.set_title('Correlation Matrix: Service Ratings + Satisfaction', fontweight='bold')
plt.tight_layout()
plt.savefig('assets/charts/05_correlation_heatmap.png', bbox_inches='tight')
plt.show()

print("\nTop correlations with Satisfaction_Binary:")
sat_corr = corr_matrix['Satisfaction_Binary'].drop('Satisfaction_Binary').sort_values(ascending=False)
print(sat_corr.round(3))


### 7.6 Delay Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, col in zip(axes, ['Departure Delay', 'Arrival Delay']):
    data_sat = df[df['Satisfaction'] == 'Satisfied'][col]
    data_dis = df[df['Satisfaction'] == 'Neutral or Dissatisfied'][col]
    ax.hist(data_sat.clip(upper=200), bins=40, alpha=0.6, label='Satisfied', color='#4c9ed9', density=True)
    ax.hist(data_dis.clip(upper=200), bins=40, alpha=0.6, label='Neutral/Dissatisfied', color='#e07b54', density=True)
    ax.set_title(f'{col} by Satisfaction (capped at 200 min)')
    ax.set_xlabel(f'{col} (minutes)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    med_s = data_sat.median()
    med_d = data_dis.median()
    ax.axvline(med_s, color='#4c9ed9', linestyle='--', linewidth=1.5, label=f'Median Sat: {med_s:.0f}')
    ax.axvline(med_d, color='#e07b54', linestyle='--', linewidth=1.5, label=f'Median Dis: {med_d:.0f}')

plt.suptitle('Delay Distribution by Satisfaction Level', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('assets/charts/06_delay_analysis.png', bbox_inches='tight')
plt.show()

print("Median Departure Delay (minutes):")
print(df.groupby('Satisfaction')['Departure Delay'].median())
print("\nMedian Arrival Delay (minutes):")
print(df.groupby('Satisfaction')['Arrival Delay'].median())


### 7.7 Flight Distance by Satisfaction

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for label, colour in [('Satisfied','#4c9ed9'), ('Neutral or Dissatisfied','#e07b54')]:
    subset = df[df['Satisfaction'] == label]['Flight Distance']
    ax.hist(subset, bins=50, alpha=0.6, label=label, color=colour, density=True)
ax.set_title('Flight Distance Distribution by Satisfaction')
ax.set_xlabel('Flight Distance (miles)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.savefig('assets/charts/07_flight_distance.png', bbox_inches='tight')
plt.show()

print("Median flight distance by satisfaction:")
print(df.groupby('Satisfaction')['Flight Distance'].median())


## 8. Business Question Answers

In [ ]:
sat_rate = (df['Satisfaction'] == 'Satisfied').mean() * 100

print("=" * 65)
print("BQ1: Proportion satisfied vs neutral/dissatisfied")
print(f"  Satisfied             : {sat_rate:.1f}%")
print(f"  Neutral/Dissatisfied  : {100-sat_rate:.1f}%")

print()
print("BQ2: Satisfaction by Customer Type:")
print(df.groupby('Customer Type')['Satisfaction_Binary'].mean().mul(100).round(1).to_string())

print()
print("BQ3: Satisfaction by Type of Travel:")
print(df.groupby('Type of Travel')['Satisfaction_Binary'].mean().mul(100).round(1).to_string())

print()
print("BQ4: Satisfaction by Class:")
print(df.groupby('Class')['Satisfaction_Binary'].mean().mul(100).round(1).to_string())

print()
print("BQ5: Satisfaction by Age Group:")
print(df.groupby('Age Group', observed=True)['Satisfaction_Binary'].mean().mul(100).round(1).to_string())

print()
print("BQ6: Top 5 service dims most correlated with satisfaction:")
print(sat_corr.head(5).round(3).to_string())

print()
print("BQ7: Mean delay by satisfaction:")
print(df.groupby('Satisfaction')[['Departure Delay','Arrival Delay']].mean().round(1).to_string())

print()
print("BQ8: Mean flight distance by satisfaction:")
print(df.groupby('Satisfaction')['Flight Distance'].mean().round(1).to_string())

print()
print("BQ9: Satisfaction by Gender:")
print(df.groupby('Gender')['Satisfaction_Binary'].mean().mul(100).round(1).to_string())


## 9. Passenger Segmentation Analysis

In [ ]:
# Cross-tabulation: Customer Type × Type of Travel × Satisfaction Rate
seg = df.groupby(['Customer Type', 'Type of Travel', 'Class'])['Satisfaction_Binary'].agg(
    Count='count', Satisfied_Rate='mean'
).reset_index()
seg['Satisfied_Rate'] = (seg['Satisfied_Rate'] * 100).round(1)
seg = seg.sort_values('Satisfied_Rate', ascending=False)
print("Passenger Segment Satisfaction Rates:")
print(seg.to_string(index=False))


In [ ]:
# Visualise top/bottom segments
fig, ax = plt.subplots(figsize=(11, 5))
seg['Segment'] = seg['Customer Type'] + ' | ' + seg['Type of Travel'] + ' | ' + seg['Class']
seg_sorted = seg.sort_values('Satisfied_Rate')
colours = ['#4c9ed9' if v >= 50 else '#e07b54' for v in seg_sorted['Satisfied_Rate']]
ax.barh(seg_sorted['Segment'], seg_sorted['Satisfied_Rate'], color=colours, edgecolor='white')
ax.axvline(50, color='grey', linestyle='--', linewidth=0.8)
ax.set_xlabel('Satisfaction Rate (%)')
ax.set_title('Satisfaction Rate by Passenger Segment', fontweight='bold')
for i, (_, row) in enumerate(seg_sorted.iterrows()):
    ax.text(row['Satisfied_Rate'] + 0.5, i, f"{row['Satisfied_Rate']:.1f}% (n={row['Count']:,})",
            va='center', fontsize=8)
plt.tight_layout()
plt.savefig('assets/charts/08_segment_satisfaction.png', bbox_inches='tight')
plt.show()


## 10. Data Leakage Prevention Audit

### Leakage Audit Checklist

| Variable | Leakage Risk? | Decision |
|----------|--------------|----------|
| `Satisfaction` | **TARGET** — never used as input | Excluded from features |
| `Satisfaction_Binary` | Encoded target | Excluded from features |
| `ID` | Already dropped | N/A |
| `Age Group` | Derived from `Age` | `Age` kept; `Age Group` excluded to avoid duplicate info |
| All other variables | No direct link to target | Included as features |

**Train/Test Split Strategy:**
- 80/20 stratified split (preserves target class proportions)
- All preprocessing (scaling, encoding) fit **only on training data**, applied to test data
- Sklearn `Pipeline` used to prevent any data leakage


In [ ]:
# ── 10.1 Define Feature Matrix and Target ─────────────────────────────────────
EXCLUDE = ['Satisfaction', 'Satisfaction_Binary', 'Age Group']  # target + derived/auxiliary

FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE]
print(f"Features used ({len(FEATURE_COLS)}):")
for f in FEATURE_COLS:
    print(f"  {f}")

X = df[FEATURE_COLS]
y = df['Satisfaction_Binary']

print(f"\nX shape: {X.shape}")
print(f"y distribution: {dict(y.value_counts())}")


In [ ]:
# ── 10.2 Identify Categorical and Numerical Columns ──────────────────────────
CAT_COLS = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
NUM_COLS = X.select_dtypes(include=['number']).columns.tolist()
print(f"Categorical ({len(CAT_COLS)}): {CAT_COLS}")
print(f"Numerical   ({len(NUM_COLS)}): {NUM_COLS}")


In [ ]:
# ── 10.3 Train-Test Split (stratified) ────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training set   : {X_train.shape[0]:,} rows")
print(f"Test set       : {X_test.shape[0]:,} rows")
print(f"\nTarget balance in train: {dict(y_train.value_counts(normalize=True).round(3))}")
print(f"Target balance in test : {dict(y_test.value_counts(normalize=True).round(3))}")


In [ ]:
# ── 10.4 Build Sklearn Pipeline ───────────────────────────────────────────────
# Preprocessing: OneHotEncode categoricals, StandardScale numericals
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), NUM_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_COLS)
], remainder='drop')

lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs'))
])

print("Logistic Regression Pipeline configured.")
print("\nPreprocessor steps:")
print(lr_pipeline)


## 11. Logistic Regression — Training & Evaluation

In [ ]:
# ── 11.1 Fit the model ────────────────────────────────────────────────────────
print("Training Logistic Regression... (this may take ~30-60 seconds on 100k rows)")
lr_pipeline.fit(X_train, y_train)
print("Training complete.")


In [ ]:
# ── 11.2 Predictions ─────────────────────────────────────────────────────────
y_pred = lr_pipeline.predict(X_test)
y_prob = lr_pipeline.predict_proba(X_test)[:, 1]  # probability of Satisfied

# ── 11.3 Metrics ─────────────────────────────────────────────────────────────
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)

print("=" * 45)
print("  LOGISTIC REGRESSION MODEL METRICS")
print("=" * 45)
print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  ROC-AUC   : {auc:.4f}")
print("=" * 45)


In [ ]:
# ── 11.4 Classification Report ───────────────────────────────────────────────
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred,
      target_names=['Neutral/Dissatisfied', 'Satisfied']))


In [ ]:
# ── 11.5 Confusion Matrix ────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=['Neutral/Diss.', 'Satisfied'])
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Confusion Matrix — Logistic Regression (Test Set)', fontweight='bold')
plt.tight_layout()
plt.savefig('assets/charts/09_confusion_matrix.png', bbox_inches='tight')
plt.show()

TN, FP, FN, TP = cm.ravel()
print(f"True Negatives  (Correctly identified Neutral/Diss.): {TN:,}")
print(f"False Positives (Wrongly predicted Satisfied)        : {FP:,}")
print(f"False Negatives (Wrongly predicted Neutral/Diss.)    : {FN:,}")
print(f"True Positives  (Correctly identified Satisfied)     : {TP:,}")


In [ ]:
# ── 11.6 ROC Curve ───────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='#4c9ed9', lw=2, label=f'LR (AUC = {auc:.3f})')
ax.plot([0,1],[0,1],'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Logistic Regression')
ax.legend()
plt.tight_layout()
plt.savefig('assets/charts/10_roc_curve.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── 11.7 Feature Importance (LR Coefficients) ─────────────────────────────────
# Get feature names after one-hot encoding
try:
    cat_feature_names = lr_pipeline.named_steps['preprocessor']\
        .named_transformers_['cat'].get_feature_names_out(CAT_COLS).tolist()
    all_feature_names = NUM_COLS + cat_feature_names

    coef = lr_pipeline.named_steps['classifier'].coef_[0]
    coef_df = pd.DataFrame({'Feature': all_feature_names, 'Coefficient': coef})
    coef_df['Abs_Coef'] = coef_df['Coefficient'].abs()
    coef_df_sorted = coef_df.sort_values('Abs_Coef', ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(10, 7))
    colours = ['#4c9ed9' if c >= 0 else '#e07b54' for c in coef_df_sorted['Coefficient']]
    ax.barh(coef_df_sorted['Feature'][::-1], coef_df_sorted['Coefficient'][::-1],
            color=colours[::-1], edgecolor='white')
    ax.axvline(0, color='grey', linewidth=0.8, linestyle='--')
    ax.set_title('Top 20 Logistic Regression Coefficients\n(Positive = associated with Satisfied)',
                 fontweight='bold')
    ax.set_xlabel('Coefficient Value')
    plt.tight_layout()
    plt.savefig('assets/charts/11_lr_coefficients.png', bbox_inches='tight')
    plt.show()
    print("Top 10 features by absolute coefficient:")
    print(coef_df_sorted[['Feature','Coefficient']].head(10).to_string(index=False))
except Exception as e:
    print(f"Coefficient extraction note: {e}")


## 12. Random Forest — Secondary Comparison

In [ ]:
# Build a Random Forest pipeline for comparison
rf_pipeline = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[
        ('num', StandardScaler(), NUM_COLS),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_COLS)
    ], remainder='drop')),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

print("Training Random Forest... (may take 1-2 minutes)")
rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
rf_prob = rf_pipeline.predict_proba(X_test)[:, 1]

rf_acc = accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)
rf_f1  = f1_score(y_test, rf_pred)

print("\nModel Comparison:")
print(f"{'Model':<25} {'Accuracy':>10} {'F1-Score':>10} {'ROC-AUC':>10}")
print("-" * 55)
print(f"{'Logistic Regression':<25} {acc:>10.4f} {f1:>10.4f} {auc:>10.4f}")
print(f"{'Random Forest':<25} {rf_acc:>10.4f} {rf_f1:>10.4f} {rf_auc:>10.4f}")


## 13. Business Insights & Recommendations

> All insights below are derived directly from validated analytical results.

### Insight Framework: Observation → Insight → Hypothesis → Recommendation

---

**Finding 1: Business class returning passengers show the highest satisfaction rates**


In [ ]:
# Print the top satisfaction segments
top_seg = seg.sort_values('Satisfied_Rate', ascending=False).head(5)
print("Top 5 Passenger Segments by Satisfaction Rate:")
print(top_seg[['Segment','Satisfied_Rate','Count']].to_string(index=False))
print()
print("Bottom 5 Passenger Segments by Satisfaction Rate:")
print(seg.sort_values('Satisfied_Rate').head(5)[['Segment','Satisfied_Rate','Count']].to_string(index=False))


---

### Summary of Business Recommendations

Based on the validated analysis results:

1. **Prioritise Online Boarding experience** — it shows one of the strongest associations with satisfaction; investment in digital boarding processes can yield measurable satisfaction improvements.

2. **Improve in-flight Wi-Fi and Entertainment** — service ratings in these dimensions show large gaps between satisfied and neutral/dissatisfied passengers.

3. **Focus Economy class passengers** — Economy class passengers show substantially lower satisfaction rates; targeted service improvements for this segment offer the greatest potential uplift.

4. **Target Personal-travel passengers** — this segment shows significantly lower satisfaction than Business-travel passengers; understanding their needs can unlock new satisfaction gains.

5. **Address delay management** — while delay distributions overlap between satisfaction groups, reducing extreme delays reduces a key source of passenger frustration.

6. **Retain Returning customers** — Returning customers show higher satisfaction; loyalty programmes and service consistency can protect this advantage.

> **Note:** All recommendations reflect associations found in the data. Controlled experiments (A/B tests) would be needed to establish causal impact before committing to large-scale investment.


## 14. Conclusion

This project delivered:

- A complete **data audit** identifying 393 missing `Arrival Delay` values (imputed conservatively), zero-rating service values (retained per data dictionary), and no duplicate rows
- A thorough **EDA** across satisfaction segments, service quality dimensions, and operational metrics
- **10 business questions** answered with quantitative evidence
- A **passenger segmentation** analysis across 12 natural segments
- A **data leakage-free** Logistic Regression model built with an sklearn Pipeline (80/20 stratified split)
- Full **model evaluation** including Confusion Matrix, Classification Report, ROC Curve, and LR coefficients
- **6 business recommendations** derived solely from validated results
- A **Streamlit dashboard** for interactive exploration and prediction (see `app.py`)

The primary model (Logistic Regression) was selected in alignment with the AICTE | IBM SkillsBuild internship curriculum. A Random Forest comparison was included to demonstrate the trade-off between interpretability and predictive accuracy.
